In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
!pip uninstall -y tensorflow



Found existing installation: tensorflow 2.17.1
Uninstalling tensorflow-2.17.1:
  Successfully uninstalled tensorflow-2.17.1


In [ ]:
import numpy as np
import pandas as pd
import time
import datetime
import gc
import random
from nltk.corpus import stopwords
import re

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler,random_split
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

import transformers
from transformers import BertForSequenceClassification, AdamW, BertConfig,BertTokenizer,get_linear_schedule_with_warmup

In [ ]:
df = pd.read_csv('cleaned_df_stopword.csv')


In [ ]:
df.head()

,Unnamed: 0,Author_name,topic,title,Abstract,year,Source,label,Word_Count
0,0,"Iain Carmichael, J. S. Marron",machine learning,Data Science vs. Statistics: Two Cultures?,"data science business learning data, tradition...",2017,arxiv,human,125
1,1,"Jannis Kueck, Ye Luo, Martin Spindler, Zigan Wang",machine learning,Estimation and Inference of Treatment Effects ...,empirical researchers increasingly faced rich ...,2017,arxiv,human,163
2,2,Jason Toy,machine learning,SenseNet: 3D Objects Database and Tactile Simu...,"majority artificial intelligence research, rel...",2017,arxiv,human,167
3,3,"Yu-Ren Liu, Yi-Qi Hu, Hong Qian, Chao Qian, Ya...",machine learning,ZOOpt: Toolbox for Derivative-Free Optimization,recent advances derivative free optimization a...,2017,arxiv,human,113
4,4,Abien Fred Agarap,machine learning,Towards Building an Intelligent Anti-Malware S...,effective efficient mitigation malware long ti...,2017,arxiv,human,189


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43264 entries, 0 to 43263
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Unnamed: 0   43264 non-null  int64 
 1   Author_name  43264 non-null  object
 2   topic        43264 non-null  object
 3   title        43264 non-null  object
 4   Abstract     43264 non-null  object
 5   year         43264 non-null  int64 
 6   Source       43264 non-null  object
 7   label        43264 non-null  object
 8   Word_Count   43264 non-null  int64 
dtypes: int64(3), object(6)
memory usage: 3.0+ MB


In [ ]:
filtered_df = df[~df.apply(lambda row: row.astype(str).str.contains('human', case=False).any(), axis=1)]


In [ ]:
filtered_df.head()

,Unnamed: 0,Author_name,topic,title,Abstract,year,Source,label,Word_Count
14533,14539,gemini,machine learning,Data Science vs. Statistics: Two Cultures?,"data science, often understood broader, task d...",2024,gemini,AI,78
14534,14540,gemini,machine learning,Estimation and Inference of Treatment Effects ...,empirical research increasingly deals large da...,2024,gemini,AI,102
14536,14542,gemini,machine learning,ZOOpt: Toolbox for Derivative-Free Optimization,zoopt zeroth order optimization derivative fre...,2024,gemini,AI,84
14537,14543,gemini,machine learning,Towards Building an Intelligent Anti-Malware S...,effective malware mitigation longstanding chal...,2024,gemini,AI,101
14538,14544,gemini,machine learning,Learning Relevant Features of Data with Multi-...,"inspired coarse graining approaches physics, p...",2024,gemini,AI,67


In [ ]:
filtered_df.describe()

,Unnamed: 0,year,Word_Count
count,25865.000000,25865.0,25865.000000
mean,28959.105161,2024.0,170.930253
std,8305.489546,0.0,52.436123
min,14539.000000,2024.0,6.000000
25%,21831.000000,2024.0,131.000000
50%,28870.000000,2024.0,184.000000
75%,36153.000000,2024.0,210.000000
max,43297.000000,2024.0,332.000000


In [ ]:
filtered_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 25865 entries, 14533 to 43263
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Unnamed: 0   25865 non-null  int64 
 1   Author_name  25865 non-null  object
 2   topic        25865 non-null  object
 3   title        25865 non-null  object
 4   Abstract     25865 non-null  object
 5   year         25865 non-null  int64 
 6   Source       25865 non-null  object
 7   label        25865 non-null  object
 8   Word_Count   25865 non-null  int64 
dtypes: int64(3), object(6)
memory usage: 2.0+ MB


In [ ]:
abstracts = filtered_df.Abstract.values
Source = filtered_df.Source.values


In [ ]:
len(Source)

25865

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased', do_lower_case=True)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
print(' Original: ', abstracts[0])

# Print the sentence split into tokens.
print('Tokenized: ', tokenizer.tokenize(abstracts[0]))

# Print the sentence mapped to token ids.
print('Token IDs: ', tokenizer.convert_tokens_to_ids(tokenizer.tokenize(abstracts[0])))


 Original:  data science, often understood broader, task driven, computationally oriented version statistics, roots traditional discipline statistics paper advocates big tent view data analysis, encompassing evolving approaches modern data analysis, exploratory analysis, machine learning, reproducibility, computation, communication, role theory, within broader context statistics examines trends impact future statistics, highlighting promising directions communication, education, research
Tokenized:  ['data', 'science', ',', 'often', 'understood', 'broader', ',', 'task', 'driven', ',', 'computational', '##ly', 'oriented', 'version', 'statistics', ',', 'roots', 'traditional', 'discipline', 'statistics', 'paper', 'advocates', 'big', 'tent', 'view', 'data', 'analysis', ',', 'encompassing', 'evolving', 'approaches', 'modern', 'data', 'analysis', ',', 'ex', '##pl', '##ora', '##tory', 'analysis', ',', 'machine', 'learning', ',', 'rep', '##rod', '##uc', '##ibility', ',', 'computation', ',', 'c

In [ ]:
###########
input_ids = []
attention_masks = []

# For every abstratc
for abstract in abstracts:
    # `encode_plus` will:
    #   (1) Tokenize the sentence.
    #   (2) Prepend the `[CLS]` token to the start.
    #   (3) Append the `[SEP]` token to the end.
    #   (4) Map tokens to their IDs.
    #   (5) Pad or truncate the sentence to `max_length`
    #   (6) Create attention masks for [PAD] tokens.
    encoded_dict = tokenizer.encode_plus(
                        abstract                  ,    # Sentence to encode.
                        add_special_tokens = True ,    # Add '[CLS]' and '[SEP]'
                        max_length = 512          ,    # Pad & truncate all sentences.
                        pad_to_max_length = True,
                        return_attention_mask = True,  # Construct attn. masks.
                        return_tensors = 'pt',         # Return pytorch tensors.
                   )

    # Add the encoded sentence to the list.
    input_ids.append(encoded_dict['input_ids'])

    # And its attention mask (simply differentiates padding from non-padding).
    attention_masks.append(encoded_dict['attention_mask'])

# Convert the lists into tensors.
input_ids = torch.cat(input_ids, dim=0)
attention_masks = torch.cat(attention_masks, dim=0)


/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:2673: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, or use `padding='max_length'` to pad to a max length. In this case, you can give a specific length with `max_length` (e.g. `max_length=45`) or leave max_length to None to pad to the maximal input size of the model (e.g. 512 for Bert).
  warnings.warn(


In [ ]:
df['Source'] = df['Source'].str.replace('Llama33', 'Llama3')

# Save the updated dataset back to a new CSV file
# df.to_csv("cleaned_df_stopword.csv", index=False)

In [ ]:



from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

source = le.fit_transform(filtered_df['Source'])
# source = df_cleaned.source.values
source = torch.tensor(source)

In [ ]:
filtered_df['Source'].value_counts()

,count
Source,
Llama3,12967
gemini,12898


In [ ]:
# torch.save({"input_ids": input_ids, "attention_masks": attention_masks, "Source": source}, "bert_input_only_llm.pt")
#

# do run from here after uploading bert_input_only_llm.pt file


In [ ]:
encoded_data = torch.load('/content/drive/MyDrive/model_req/bert_input_only_llm.pt')

input_ids = encoded_data['input_ids']
attention_masks = encoded_data['attention_masks']
source = encoded_data['Source']

print("Tensors loaded successfully!")

<ipython-input-20-b75c3de5677e>:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  encoded_data = torch.load('/content/drive/MyDrive/model_req/bert_input_only_llm.pt')


Tensors loaded successfully!


In [ ]:


dataset = TensorDataset(input_ids, attention_masks, source)

# Create a 75-15-10 train-validation-testing split.
# Calculate the number of samples to include in each set.
train_size = int(0.75 * len(dataset))
val_size = int(0.15 * len(dataset))
test_size = int(0.1 * len(dataset))+2 # due to conversion to integer


In [ ]:
len(dataset)

25865

In [ ]:
train_size+val_size+test_size

25865

In [ ]:
# Divide the dataset by randomly selecting samples.
# train_dataset, val_dataset = random_split(dataset, [train_size, val_size])
train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size])


print('{:>5,} training samples'.format(train_size))
print('{:>5,} validation samples'.format(val_size))
print('{:>5,} testing samples'.format(test_size))


19,398 training samples
3,879 validation samples
2,588 testing samples


In [ ]:
# The DataLoader needs to know our batch size for training, so we specify it
# here. For fine-tuning BERT on a specific task, the authors recommend a batch
# size of 16 or 32.
batch_size = 16

# Create the DataLoaders for our training and validation sets.
# We'll take training samples in random order.
train_dataloader = DataLoader(
            train_dataset,  # The training samples.
            sampler = RandomSampler(train_dataset), # Select batches randomly
            batch_size = batch_size # Trains with this batch size.
        )

# For validation the order doesn't matter, so we'll just read them sequentially.
validation_dataloader = DataLoader(
            val_dataset, # The validation samples.
            sampler = RandomSampler(val_dataset), # Pull out batches sequentially.
            batch_size = batch_size # Evaluate with this batch size.
        )

# For testing the order doesn't matter, so we'll just read them sequentially.
testing_dataloader = DataLoader(
            test_dataset,      # The validation samples.
            sampler = RandomSampler(test_dataset), # Pull out batches sequentially.
            batch_size = batch_size # Evaluate with this batch size.
        )

In [ ]:
# Load BertForSequenceClassification, the pretrained BERT model with a single
# linear classification layer on top.
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased", # Use the 12-layer BERT model, with an uncased vocab.
    num_labels = 2, # The number of output labels--2 for binary classification.
                    # You can increase this for multi-class tasks.
    output_attentions = True, # Whether the model returns attentions weights.
    output_hidden_states = True, # Whether the model returns all hidden-states.
)

# if device == "cuda:0":
# # Tell pytorch to run this model on the GPU.
#     model = model.cuda()
model = model.to("cpu")



/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
optimizer = AdamW(model.parameters(),
                  lr = 2e-5, # args.learning_rate - default is 5e-5, our notebook had 2e-5
                  eps = 1e-8 # args.adam_epsilon  - default is 1e-8.
                )


/usr/local/lib/python3.11/dist-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


In [ ]:


epochs = 4

# Total number of training steps is [number of batches] x [number of epochs].
# (Note that this is not the same as the number of training samples).
total_steps = len(train_dataloader) * epochs
total_steps

4852

In [ ]:



# Create the learning rate scheduler.
scheduler = get_linear_schedule_with_warmup(optimizer,
                                            num_warmup_steps = 0, # Default value in run_glue.py
                                            num_training_steps = total_steps)



In [ ]:

# Function to calculate the accuracy of our predictions vs labels
def flat_accuracy(preds, labels):
    pred_flat = np.argmax(preds, axis=1).flatten()
    labels_flat = labels.flatten()
    return np.sum(pred_flat == labels_flat) / len(labels_flat)


def format_time(elapsed):
    '''
    Takes a time in seconds and returns a string hh:mm:ss
    '''
    # Round to the nearest second.
    elapsed_rounded = int(round((elapsed)))
    # Format as hh:mm:ss
    return str(datetime.timedelta(seconds=elapsed_rounded))



In [ ]:
seed_val = 42
random.seed(seed_val)
np.random.seed(seed_val)
torch.manual_seed(seed_val)
torch.cuda.manual_seed_all(seed_val)
training_stats = []
device = "cuda"


In [ ]:
# # Measure the total training time for the whole run.
# total_t0 = time.time()

# # For each epoch...
# for epoch_i in range(0, epochs):

#     # ========================================
#     #               Training
#     # ========================================
#     # Perform one full pass over the training set.
#     print("")
#     print('======== Epoch {:} / {:} ========'.format(epoch_i + 1, epochs))
#     print('Training...')
#     # Measure how long the training epoch takes.
#     t0 = time.time()
#     total_train_loss = 0
#     model.train()
#     for step, batch in enumerate(train_dataloader):
#         # Unpack this training batch from our dataloader.
#         #
#         #  As we unpack the batch, we'll also copy each tensor to the device using the
#         # `to` method.
#         #
#         # `batch` contains three pytorch tensors:
#         #   [0]: input ids
#         #   [1]: attention masks
#         #   [2]: labels
#         print(step) if step % 40 == 0 else None
#         b_input_ids  = batch[0].to(device)
#         b_input_mask = batch[1].to(device)
#         b_labels     = batch[2].to(device)
#         optimizer.zero_grad()
#         output = model(b_input_ids,
#                              token_type_ids=None,
#                              attention_mask=b_input_mask,
#                              labels=b_labels)
#         loss = output.loss
#         total_train_loss += loss.item()
#         # Perform a backward pass to calculate the gradients.
#         loss.backward()
#         # Clip the norm of the gradients to 1.0.
#         # This is to help prevent the "exploding gradients" problem.
#         torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
#         # Update parameters and take a step using the computed gradient.
#         # The optimizer dictates the "update rule"--how the parameters are
#         # modified based on their gradients, the learning rate, etc.
#         optimizer.step()
#         # Update the learning rate.
#         scheduler.step()

#     # Calculate the average loss over all of the batches.
#     avg_train_loss = total_train_loss / len(train_dataloader)

#     # Measure how long this epoch took.
#     training_time = format_time(time.time() - t0)
#     print("")
#     print("  Average training loss: {0:.2f}".format(avg_train_loss))
#     print("  Training epoch took: {:}".format(training_time))
#     # ========================================
#     #               Validation
#     # ========================================
#     # After the completion of each training epoch, measure our performance on
#     # our validation set.
#     print("")
#     print("Running Validation...")
#     t0 = time.time()
#     # Put the model in evaluation mode--the dropout layers behave differently
#     # during evaluation.
#     model.eval()
#     # Tracking variables
#     total_eval_accuracy = 0
#     best_eval_accuracy = 0
#     total_eval_loss = 0
#     nb_eval_steps = 0
#     # Evaluate data for one epoch
#     for batch in validation_dataloader:
#         b_input_ids = batch[0].to(device)
#         b_input_mask = batch[1].to(device)
#         b_labels = batch[2].to(device)
#         # Tell pytorch not to bother with constructing the compute graph during
#         # the forward pass, since this is only needed for backprop (training).
#         with torch.no_grad():
#             output= model(b_input_ids,
#                                    token_type_ids=None,
#                                    attention_mask=b_input_mask,
#                                    labels=b_labels)
#         loss = output.loss
#         total_eval_loss += loss.item()
#         # Move logits and labels to CPU if we are using GPU
#         logits = output.logits
#         logits = logits.detach().cpu().numpy()
#         label_ids = b_labels.to('cpu').numpy()
#         # Calculate the accuracy for this batch of test sentences, and
#         # accumulate it over all batches.
#         total_eval_accuracy += flat_accuracy(logits, label_ids)
#     # Report the final accuracy for this validation run.
#     avg_val_accuracy = total_eval_accuracy / len(validation_dataloader)
#     print("  Accuracy: {0:.2f}".format(avg_val_accuracy))
#     # Calculate the average loss over all of the batches.
#     avg_val_loss = total_eval_loss / len(validation_dataloader)
#     # Measure how long the validation run took.
#     validation_time = format_time(time.time() - t0)
#     if avg_val_accuracy > best_eval_accuracy:
#         torch.save(model, 'bert_model')
#         best_eval_accuracy = avg_val_accuracy
#     #print("  Validation Loss: {0:.2f}".format(avg_val_loss))
#     #print("  Validation took: {:}".format(validation_time))
#     # Record all statistics from this epoch.
#     training_stats.append(
#         {
#             'epoch': epoch_i + 1,
#             'Training Loss': avg_train_loss,
#             'Valid. Loss': avg_val_loss,
#             'Valid. Accur.': avg_val_accuracy,
#             'Training Time': training_time,
#             'Validation Time': validation_time
#         }
#     )
# print("")
# print("Training complete!")

# print("Total training took {:} (h:mm:ss)".format(format_time(time.time()-total_t0)))

In [ ]:
import os
import torch

# Define checkpoint file
checkpoint_path = "/content/drive/MyDrive/model_req/llm_model_checkpoint.pth"

# Check if a checkpoint exists
start_epoch = 0
start_step = 0
best_eval_accuracy = 0

if os.path.exists(checkpoint_path):
    print("Loading checkpoint...")
    checkpoint = torch.load(checkpoint_path)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    start_step = checkpoint['step'] + 1
    best_eval_accuracy = checkpoint['best_eval_accuracy']
    print(f"Resuming training from epoch {start_epoch}, step {start_step}.")

# Measure the total training time for the whole run.
total_t0 = time.time()

# For each epoch...
for epoch_i in range(start_epoch, epochs):
    print("")
    print('======== Epoch {:} / {:} ========'.format(epoch_i + 1, epochs))
    print('Training...')
    t0 = time.time()
    total_train_loss = 0
    model.train()

    # Iterate over the training steps
    for step, batch in enumerate(train_dataloader):
        # Resume training from the last saved step
        if epoch_i == start_epoch and step < start_step:
            continue

        # Unpack the training batch
        b_input_ids = batch[0].to(device)
        b_input_mask = batch[1].to(device)
        b_labels = batch[2].to(device)

        optimizer.zero_grad()
        output = model(
            b_input_ids,
            token_type_ids=None,
            attention_mask=b_input_mask,
            labels=b_labels
        )
        loss = output.loss
        total_train_loss += loss.item()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        print(step)

        # Save checkpoint after every step


        # Log progress
        if step % 40 == 0:
            print(f"  Step {step}/{len(train_dataloader)}")
            torch.save({
            'epoch': epoch_i,
            'step': step,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_eval_accuracy': best_eval_accuracy
        }, checkpoint_path)


    avg_train_loss = total_train_loss / len(train_dataloader)
    training_time = format_time(time.time() - t0)
    print("")
    print(f"  Average training loss: {avg_train_loss:.2f}")
    print(f"  Training epoch took: {training_time}")

    # ========================================
    #               Validation
    # ========================================
    print("")
    print("Running Validation...")
    t0 = time.time()
    model.eval()
    total_eval_accuracy = 0
    total_eval_loss = 0

    for batch in validation_dataloader:
        b_input_ids = batch[0].to(device)
        b_input_mask = batch[1].to(device)
        b_labels = batch[2].to(device)

        with torch.no_grad():
            output = model(
                b_input_ids,
                token_type_ids=None,
                attention_mask=b_input_mask,
                labels=b_labels
            )

        loss = output.loss
        total_eval_loss += loss.item()
        logits = output.logits.detach().cpu().numpy()
        label_ids = b_labels.to('cpu').numpy()
        total_eval_accuracy += flat_accuracy(logits, label_ids)

    avg_val_accuracy = total_eval_accuracy / len(validation_dataloader)
    avg_val_loss = total_eval_loss / len(validation_dataloader)
    validation_time = format_time(time.time() - t0)

    if avg_val_accuracy > best_eval_accuracy:
        torch.save(model.state_dict(), '/content/drive/MyDrive/model_req/bert_model_llm_only__2.pth')
        best_eval_accuracy = avg_val_accuracy

    print(f"  Validation Accuracy: {avg_val_accuracy:.2f}")
    print(f"  Validation took: {validation_time}")

    training_stats.append(
        {
            'epoch': epoch_i + 1,
            'Training Loss': avg_train_loss,
            'Valid. Loss': avg_val_loss,
            'Valid. Accur.': avg_val_accuracy,
            'Training Time': training_time,
            'Validation Time': validation_time
        }
    )

print("")
print("Training complete!")
print(f"Total training took {format_time(time.time() - total_t0)} (h:mm:ss)")


Loading checkpoint...


<ipython-input-15-bc84ede22d8f>:14: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path)


Resuming training from epoch 2, step 1201.

======== Epoch 3 / 4 ========
Training...
1201
1202
1203
1204
1205
1206
1207
1208
1209
1210
1211
1212

  Average training loss: 0.00
  Training epoch took: 0:00:16

Running Validation...
  Validation Accuracy: 0.99
  Validation took: 0:02:07

======== Epoch 4 / 4 ========
Training...
0
  Step 0/1213


KeyboardInterrupt: 

In [ ]:
from transformers import BertForSequenceClassification

# Load the saved model
model.load_state_dict(torch.load('/content/drive/MyDrive/model_req/bert_model_llm_only__2.pth'))
model.to(device)
model.eval()

<ipython-input-16-52e18af277e1>:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('/content/drive/MyDrive/model_req/bert_model_llm_only__2.pt

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score

predictions = []
true_labels = []  # To store ground truth labels

for batch in testing_dataloader:
    b_input_ids = batch[0].to(device)
    b_input_mask = batch[1].to(device)
    b_labels = batch[2].to(device)  # Assuming labels are at index 2

    with torch.no_grad():
        outputs = model(b_input_ids, attention_mask=b_input_mask)
        logits = outputs.logits
        logits = logits.detach().cpu().numpy()
        label_ids = b_labels.cpu().numpy()

        # Store predictions and true labels
        pred_flat = np.argmax(logits, axis=1).flatten()
        predictions.extend(pred_flat)
        true_labels.extend(label_ids)

# Calculate accuracy
accuracy = accuracy_score(true_labels, predictions)
print(f"Accuracy: {accuracy * 100:.2f}%")


Accuracy: 99.46%


In [ ]:
import sklearn.metrics as metrics

confusion_matrix = metrics.confusion_matrix(true_labels, predictions)
print(confusion_matrix)

[[1307    1]
 [  13 1267]]


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

confusion_matrix = metrics.confusion_matrix(true_labels, predictions)
plt.figure(figsize=(10, 8))

<Figure size 1000x800 with 0 Axes>

<Figure size 1000x800 with 0 Axes>

In [ ]:
# prompt: draw roc graph of bert model

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc

# Assuming 'predictions' and 'true_labels' are already defined from your previous code

# Calculate predicted probabilities (for binary classification)
# If you have probabilities directly, skip this step
pred_probs = model(torch.tensor(b_input_ids).to(device), attention_mask=torch.tensor(b_input_mask).to(device)).logits.softmax(dim=1).detach().cpu().numpy()[:,1]


fpr, tpr, thresholds = roc_curve(true_labels, pred_probs)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
plt.show()

# CASCADED MODEL


In [ ]:
# Load BertForSequenceClassification, the pretrained BERT model with a single
# linear classification layer on top.
model_ai_hum = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased", # Use the 12-layer BERT model, with an uncased vocab.
    num_labels = 2, # The number of output labels--2 for binary classification.
                    # You can increase this for multi-class tasks.
    output_attentions = True, # Whether the model returns attentions weights.
    output_hidden_states = True, # Whether the model returns all hidden-states.
)
model_llm = model_ai_hum
# if device == "cuda:0":
# # Tell pytorch to run this model on the GPU.
#     model = model.cuda()

# model = model.to("cuda")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
from transformers import BertForSequenceClassification

# Load the saved model
model_ai_hum = torch.load('/content/drive/MyDrive/model_req/bert_model', map_location = "cpu")

# Load the saved model
model_llm.load_state_dict(torch.load('/content/drive/MyDrive/model_req/bert_model_llm_only__2.pth', map_location = "cpu"))



<ipython-input-18-c9863e38b3e8>:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_ai_hum = torch.load('/content/drive/MyDrive/model_req/bert_model', map_location = "cpu

<All keys matched successfully>

In [ ]:
input_ids

tensor([[  101,  2951,  2671,  ...,     0,     0,     0],
        [  101, 17537,  2470,  ...,     0,     0,     0],
        [  101,  9201, 13876,  ...,     0,     0,     0],
        ...,
        [  101,  2817,  7534,  ...,     0,     0,     0],
        [  101,  2470,  3259,  ...,     0,     0,     0],
        [  101,  2817,  7534,  ...,     0,     0,     0]])

In [ ]:
df.values[2000]

array([2000,
       'Jan Vykopal, Radek Ošlejšek, Karolína Burská, Kristína Zákopčanová',
       'Cybersecurity',
       'Timely Feedback in Unstructured Cybersecurity Exercises',
       'cyber defence exercises intensive, hands learning events teams professionals gain develop skills successfully prevent respond cyber attacks exercises mimic real life, routine operation organization attacked unknown offender teams learners receive limited immediate feedback instructors exercise usually see scoreboard showing aggregated gain loss points particular tasks depth analysis learners actions requires considerable human effort, results days weeks delay intensive experience thus followed proper feedback facilitating actual learning, diminishes effect exercise initial work, investigate provide valuable feedback learners right exercise without unnecessary delay based scoring system cyber defence exercise, developed new feedback tool presents interactive, personalized timeline exercise events deplo

In [ ]:
abstract = df.values[39000]
abstract

array([39034, 'Llama3', 'Climate Change',
       'A Theory-independent Way of Unambiguous Detection of Wino-like particles at LHC',
       'discovering characterizing wino like particles, hypothetical supersymmetric partners standard model w boson, remains challenging task due complexity interactions competing background processes large hadron collider lhc theoretical work, propose novel strategy unambiguous detection wino like particles independent preconceived theoretical framework focusing wino unique mass degenerate two body decay feature, explore innovative signature based approach wherein single event topology characterizing boosted lepton jets exploited enables identification higgsino mediated venustran decay mode separates clean wino candidates irreducible standard model backgrounds elegant properties higgs boson mediated vertex implementing approach refined boosted object level within context free analysis lhc events necessitates well suited advances key algorithms focusing ma

In [ ]:
import numpy as np
from transformers import AutoTokenizer
import torch

abstract = df["Abstract"].values[39000]


encoded_input = tokenizer.encode_plus(
    abstract,
    add_special_tokens=True,
    max_length=512,
    padding="max_length",
    truncation=True,
    return_tensors="pt"
)

input_ids = encoded_input["input_ids"][:2].to("cpu")
attention_mask = encoded_input["attention_mask"][:2].to("cpu")

with torch.no_grad():
    outputs = model_ai_hum(input_ids, attention_mask=attention_mask)
    logits = outputs.logits

predicted_label = np.argmax(logits.cpu().numpy(), axis=1).item()

if predicted_label == 1:
    predicted_label = "Human"
    print(f"Predicted label: {predicted_label}")

else:
    predicted_label = "AI"
    print(f"Predicted label: {predicted_label}")

    with torch.no_grad():
        outputs_llm = model_llm(input_ids, attention_mask=attention_mask)
        logits_llm = outputs_llm.logits
        predicted_label_llm = np.argmax(logits_llm.cpu().numpy(), axis=1).item()

    if predicted_label_llm == 0:
        predicted_label_llm = "LLAMA"
    else:
        predicted_label_llm = "Gemini"

    print(f"Predicted label (LLM): {predicted_label_llm}")

Predicted label: AI
Predicted label (LLM): LLAMA
